# Correlation & Feature Engineering
In this notebook, we will transform our data into a format suitable for machine learning and create new features that might help our model's performance.

Key tasks:
1. **Encoding Categorical Variables:** Converting text to numbers.
2. **Correlation Matrix:** Identifying which variables are most linked to Churn.
3. **Feature Engineering:** Creating `Tenure Cohorts` to see if customer age groups behave differently.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

sns.set_theme(style="whitegrid")
%matplotlib inline

# Load and clean
df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(inplace=True)
df.drop(columns=['customerID'], inplace=True)

print("Data loaded and base cleaning done.")

## 1. Feature Engineering: Tenure Cohorts
Numeric tenure is great, but sometimes grouping customers into buckets (e.g., 0-12 months) reveals clearer patterns.

In [ ]:
def map_tenure(tenure):
    if tenure <= 12:
        return '0-12m'
    elif tenure <= 24:
        return '12-24m'
    elif tenure <= 48:
        return '24-48m'
    elif tenure <= 60:
        return '48-60m'
    else:
        return '>60m'

df['TenureCohort'] = df['tenure'].apply(map_tenure)

plt.figure(figsize=(10, 5))
sns.countplot(data=df, x='TenureCohort', hue='Churn', palette='Set2', order=['0-12m', '12-24m', '24-48m', '48-60m', '>60m'])
plt.title('Churn by Tenure Cohort')
plt.show()

## 2. Encoding for Correlation Analysis
To calculate correlation, we need numbers. We'll use Label Encoding for binary features and One-Hot Encoding for multi-category features.

In [ ]:
# Copy for correlation works
df_encoded = df.copy()

# Binary encoding for Churn
df_encoded['Churn'] = df_encoded['Churn'].map({'Yes': 1, 'No': 0})

# One-Hot Encoding for all categorical variables
df_encoded = pd.get_dummies(df_encoded)

print("Shape after One-Hot Encoding:", df_encoded.shape)
df_encoded.head()

## 3. Correlation Heatmap
Let's see which features have the strongest positive or negative correlation with `Churn`.

In [ ]:
plt.figure(figsize=(15, 10))
churn_corr = df_encoded.corr()['Churn'].sort_values(ascending=False)
sns.barplot(x=churn_corr.values, y=churn_corr.index, palette='viridis')
plt.title('Correlation of Features with Churn')
plt.show()

**Observations:**
> *Which features are at the top? (High Churn risk) Which are at the bottom? (Retention drivers)*

## 4. Multicollinearity Check
Does `TotalCharges` just repeat what `tenure` and `MonthlyCharges` tell us?

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df[['tenure', 'MonthlyCharges', 'TotalCharges']].corr(), annot=True, cmap='summer')
plt.title('Correlation between Numeric Features')
plt.show()